# Sieci rekurencyjne

W prezentacji skupiamy się na **przetwarzaniu języka naturalnego (NLP)**. Tutaj zobaczymy RNN „od środka”, nauczymy niewielki model klasyfikować polskie opinie i porównamy `nn.RNN`, `nn.LSTM` oraz `nn.GRU`. **Opcjonalnie** na końcu spróbujemy przewidywania szeregu czasowego.

**Wymagania:** Python 3.10+, `torch`, `numpy`, `matplotlib` (w Google Colab zazwyczaj dostępne). Nie potrzeba GPU.

> Uwaga: teksty w naszym zbiorze są **sztucznie skonstruowane** i bardzo proste. Wysoki wynik klasyfikacji oznacza tu zrozumienie reguł małego zadania, **nie** gotowość do analizy prawdziwych recenzji.

## 0. Przygotowanie środowiska

Najpierw ustawiamy ziarno losowości, żeby łatwiej było powtórzyć wyniki. Ograniczamy liczbę wątków procesora, by małe ćwiczenie działało płynnie także na laptopach.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)  # Małe modele często działają szybciej bez wielu wątków.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Urządzenie:", DEVICE)

## 1. Sekwencja to nie zbiór elementów

Porównajmy „pies goni kota” i „kota goni pies”. Mimo podobnych słów znaczenie jest inne, ponieważ ważne jest **ich położenie**. RNN przetwarza słowa po kolei i aktualizuje swój **stan ukryty** – wektor przechowujący informację o kontekście.

In [ ]:
zdanie_a = "pies goni kota".split()
zdanie_b = "kota goni pies".split()
print("Słowa A:", zdanie_a)
print("Słowa B:", zdanie_b)
print("Ten sam zbiór tokenów?", set(zdanie_a) == set(zdanie_b))
print("Ta sama sekwencja?", zdanie_a == zdanie_b)

# Dla modelu językowego wejściem są tokeny od początku zdania.
# Celem jest zawsze KOLEJNY token. To przesunięcie o jedną pozycję.
tokeny = ["<start>", "ala", "ma", "kota", "<eos>"]
print("Wejście modelu językowego:", tokeny[:-1])
print("Poprawne następne tokeny: ", tokeny[1:])

## 2. Najprostsza pamięć rekurencyjna

Prawdziwe RNN liczy (dla wektorów)

$h_t = \tanh(W_xx_t + W_hh_{t-1} + b).$

- `x_t`: nowa informacja (np. aktualne słowo),
- `h_{t-1}`: pamięć po poprzednim kroku,
- `h_t`: nowa pamięć,
- **te same wagi** `W_x` i `W_h` są stosowane na każdym kroku.

Najpierw uprośćmy wszystko do **jednej liczby**, by było widać działanie. To zabawkowy model – nie uczymy go jeszcze.

In [ ]:
wejscia = [1.0, 0.0, 0.0, 0.0, 0.0]
w_wejscie = 1.0
w_pamiec = 0.8
h = torch.tensor(0.0)

stany = []
for t, x in enumerate(wejscia):
    # Nowa pamięć zależy od bieżącej liczby i poprzedniej pamięci.
    h = torch.tanh(w_wejscie * torch.tensor(x) + w_pamiec * h)
    stany.append(h.item())
    print(f"Krok {t}: x={x:.0f}, stan ukryty h={h.item():.3f}")

plt.figure(figsize=(7, 3))
plt.plot(range(len(stany)), stany, marker="o")
plt.title("Prosta pamięć: informacja słabnie, ale nie znika natychmiast")
plt.xlabel("Krok w sekwencji")
plt.ylabel("Wartość stanu ukrytego")
plt.grid(alpha=0.3)
plt.show()

### Minićwiczenie 1

Zmień `w_pamiec` z `0.8` na `0.2`, a potem na `1.0`. Jak szybko ginie informacja o pierwszym wejściu? Dlaczego RNN może mieć problem z zapamiętaniem początku długiego zdania?

## 3. BPTT: uczenie przez czas i problem gradientów

Podczas uczenia **Backpropagation Through Time (BPTT)** „rozwija” sieć na kolejne kroki, oblicza stratę i propaguje gradient od późniejszych kroków w stronę wcześniejszych. Jeżeli wiele razy mnożymy przez czynnik mniejszy od 1, gradient może **zanikać**; jeśli przez czynnik większy od 1 – może **eksplodować**.

Poniższy eksperyment to **liniowe uproszczenie** `h_t = w · h_(t−1)` (bez `tanh`). Pozwala dokładnie zobaczyć iloczyn `w^T`; w prawdziwym RNN na gradienty wpływają także macierze wag i pochodne funkcji aktywacji.

In [ ]:
dlugosci = np.arange(1, 26)
plt.figure(figsize=(8, 4))
for w in [0.5, 1.0, 1.3]:
    # Pochodna stanu końcowego względem stanu początkowego to w^T.
    przeplyw_gradientu = w ** dlugosci
    plt.semilogy(dlugosci, przeplyw_gradientu, marker=".", label=f"w = {w}")
plt.xlabel("Liczba kroków T")
plt.ylabel("|∂h_T / ∂h_0| (skala logarytmiczna)")
plt.title("Dlaczego gradient może znikać lub rosnąć?")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# PyTorch automatycznie wyznacza pochodne za pomocą autograd.
w = torch.tensor(0.5, requires_grad=True)
h0 = torch.tensor(1.0, requires_grad=True)
h = h0
for _ in range(12):
    h = w * h
h.backward()
print("h_12 =", round(h.item(), 8))
print("Gradient ∂h_12/∂h_0 =", round(h0.grad.item(), 8))
print("Wynik analityczny 0.5^12 =", 0.5 ** 12)

# W prawdziwym treningu eksplozji gradientów przeciwdziała np. clip_grad_norm_.
# Funkcję wywołujemy PO loss.backward(), a PRZED optimizer.step().

### Minićwiczenie 2

Zmień `w` na `1.3`. Co stanie się z pochodną? Wyjaśnij, dlaczego `clip_grad_norm_` ogranicza problem **eksplodujących**, ale sam nie naprawia **zanikających** gradientów.

## 4. Zadanie NLP: czy opinia jest pozytywna?

Zbudujemy niewielki zbiór polskich opinii. Występują w nim oceny dodatnie i ujemne, a **słowo „nie” odwraca etykietę**: „dobry” → pozytywna, „nie dobry” → negatywna. Dzięki temu model nie powinien patrzeć jedynie na obecność pozytywnego przymiotnika.

**Nie jest to realistyczny korpus językowy:** reguły są sztucznie proste, a prawdziwe zdania zawierają ironię, wieloznaczność i bardziej skomplikowane zaprzeczenia.

In [ ]:
pochwaly = ["dobry", "świetny", "ciekawy", "udany", "zabawny", "wspaniały"]
krytyka = ["zły", "słaby", "nudny", "kiepski", "fatalny", "męczący"]
obiekty = ["film", "serial", "występ", "odcinek"]
szablony = [
    "ten {obiekt} był {ocena}",
    "uważam że {obiekt} był {ocena}",
    "moim zdaniem {obiekt} był {ocena}",
    "wczoraj obejrzałem {obiekt} i był {ocena}",
]

przyklady = []
for przymiotniki, bazowa_etykieta in [(pochwaly, 1), (krytyka, 0)]:
    for przymiotnik in przymiotniki:
        for obiekt in obiekty:
            for szablon in szablony:
                for negacja in [False, True]:
                    ocena = ("nie " if negacja else "") + przymiotnik
                    tekst = szablon.format(obiekt=obiekt, ocena=ocena)
                    etykieta = 1 - bazowa_etykieta if negacja else bazowa_etykieta
                    przyklady.append((tekst, etykieta))

rng = random.Random(SEED)
rng.shuffle(przyklady)
liczba_treningowych = int(0.8 * len(przyklady))
trening = przyklady[:liczba_treningowych]
test = przyklady[liczba_treningowych:]
print(f"Zbiór: {len(trening)} treningowych, {len(test)} testowych")
for tekst, y in trening[:6]:
    print(f"{tekst:50s} → {'pozytywna' if y else 'negatywna'}")

### Jak sieć widzi słowa?

1. Dzielimy zdanie na tokeny (tu – w uproszczeniu – według spacji).
2. Każdemu słowu przypisujemy numer (`token ID`).
3. `nn.Embedding` zamienia numer na **wektor liczb** uczony razem z całą siecią.
4. Krótsze zdania dopełniamy specjalnym tokenem `<pad>`, ale klasyfikacja korzysta z **ostatniego rzeczywistego kroku**, nie z paddingu.

W bardziej rozbudowanych modelach używa się zaawansowanej tokenizacji, obsługi nowych słów i lepszych zbiorów.

In [ ]:
wszystkie_slowa = sorted({slowo for tekst, _ in przyklady for slowo in tekst.split()})
vocab = {"<pad>": 0, "<unk>": 1}
for slowo in wszystkie_slowa:
    vocab[slowo] = len(vocab)


def zakoduj(tekst):
    # Nieznane słowo otrzymuje indeks tokenu <unk>.
    return [vocab.get(slowo, vocab["<unk>"]) for slowo in tekst.lower().split()]


class OpinieDataset(Dataset):
    def __init__(self, rekordy):
        self.rekordy = rekordy

    def __len__(self):
        return len(self.rekordy)

    def __getitem__(self, idx):
        tekst, etykieta = self.rekordy[idx]
        return torch.tensor(zakoduj(tekst)), etykieta


def sklej_batch(batch):
    # W obrębie batcha dopełniamy wszystkie zdania do tej samej długości.
    sekwencje, etykiety = zip(*batch)
    dlugosci = torch.tensor([len(s) for s in sekwencje])
    maks_dl = max(dlugosci).item()
    x = torch.zeros(len(sekwencje), maks_dl, dtype=torch.long)
    for i, seq in enumerate(sekwencje):
        x[i, :len(seq)] = seq
    y = torch.tensor(etykiety, dtype=torch.long)
    return x, dlugosci, y

trening_loader = DataLoader(OpinieDataset(trening), batch_size=32,
                             shuffle=True, collate_fn=sklej_batch)
test_loader = DataLoader(OpinieDataset(test), batch_size=64,
                          shuffle=False, collate_fn=sklej_batch)

x, dlugosci, y = next(iter(trening_loader))
print("Liczba słów w słowniku:", len(vocab))
print("Kształt wejść [batch, kroki]:", tuple(x.shape))
print("Długości 5 pierwszych zdań:", dlugosci[:5].tolist())
print("Etykiety 5 pierwszych zdań:", y[:5].tolist())

## 5. Jedna sieć, trzy rodzaje pamięci: RNN, LSTM, GRU

`nn.RNN` przechowuje stan ukryty. `nn.LSTM` ma dodatkowo stan komórki (długotrwałą pamięć), a `nn.GRU` używa uproszczonego zestawu bramek. Z punktu widzenia naszego zadania wystarczy zmienić **jeden argument** konstrukcji modelu.

W tym przykładzie używamy klasyfikacji całego zdania, więc sieć przekazuje dalej wynik z ostatniego **niepustego** tokenu. Gdybyśmy przewidywali etykietę każdego słowa (np. NER), wykorzystalibyśmy wyjścia z **każdego** kroku.

In [ ]:
class KlasyfikatorSekwencji(nn.Module):
    def __init__(self, rozmiar_slownika, typ="GRU", wymiar_embeddingu=16,
                 rozmiar_stanu=32):
        super().__init__()
        self.typ = typ.upper()
        self.embedding = nn.Embedding(rozmiar_slownika, wymiar_embeddingu,
                                      padding_idx=0)
        rodzaje = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}
        if self.typ not in rodzaje:
            raise ValueError("typ musi być RNN, GRU albo LSTM")
        # batch_first=True: tensor ma kształt [liczba_zdań, długość, cechy].
        self.rekurencja = rodzaje[self.typ](wymiar_embeddingu, rozmiar_stanu,
                                            batch_first=True)
        self.klasyfikator = nn.Linear(rozmiar_stanu, 2)

    def forward(self, tokeny, dlugosci):
        wektory = self.embedding(tokeny)
        # out zawiera reprezentację każdego kroku sekwencji.
        out, _ = self.rekurencja(wektory)
        # Dla każdego zdania pobieramy jego ostatnią prawdziwą pozycję.
        indeksy = (dlugosci - 1).to(out.device)
        ostatni_stan = out[torch.arange(len(tokeny), device=out.device), indeksy]
        # Zwracamy surowe logity: CrossEntropyLoss sama liczy softmax.
        return self.klasyfikator(ostatni_stan)

probe = KlasyfikatorSekwencji(len(vocab), "GRU")
with torch.no_grad():
    logity = probe(x, dlugosci)
print("Wejście:", tuple(x.shape))
print("Wyjście (dwie klasy na zdanie):", tuple(logity.shape))
print("Przykładowe prawdopodobieństwa:",
      torch.softmax(logity[:3], dim=1).round(decimals=3))

### Jak przebiega trening?

1. `model(x)`: przewidywanie etykiet dla tekstu.
2. `loss = CrossEntropyLoss(logity, y)`: policzenie błędu.
3. `loss.backward()`: **backpropagation through time** – PyTorch liczy gradienty.
4. `clip_grad_norm_(...)`: opcjonalne ograniczenie zbyt dużego gradientu.
5. `optimizer.step()`: poprawienie parametrów modelu.

Pamiętaj o `model.train()` / `model.eval()`: tutaj nie ma dropout, ale jest to dobra praktyka.

In [9]:
def ocen(model, loader):
    model.eval()
    suma_trafien, suma_przykladow, suma_strat = 0, 0, 0.0
    kryterium = nn.CrossEntropyLoss()
    with torch.no_grad():
        for tokeny, dlugosci, etykiety in loader:
            tokeny = tokeny.to(DEVICE)
            dlugosci = dlugosci.to(DEVICE)
            etykiety = etykiety.to(DEVICE)
            logity = model(tokeny, dlugosci)
            suma_strat += kryterium(logity, etykiety).item() * len(etykiety)
            suma_trafien += (logity.argmax(dim=1) == etykiety).sum().item()
            suma_przykladow += len(etykiety)
    return suma_strat / suma_przykladow, suma_trafien / suma_przykladow


def trenuj(typ="GRU", epoki=12, lr=0.01):
    # Tworzymy NOWY model, żeby porównanie rodzajów sieci było uczciwe.
    torch.manual_seed(SEED)
    model = KlasyfikatorSekwencji(len(vocab), typ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    kryterium = nn.CrossEntropyLoss()
    historia = {"strata_trening": [], "strata_test": [], "trafnosc_test": []}

    for epoka in range(epoki):
        model.train()
        suma_strat = 0.0
        for tokeny, dlugosci, etykiety in trening_loader:
            tokeny = tokeny.to(DEVICE)
            dlugosci = dlugosci.to(DEVICE)
            etykiety = etykiety.to(DEVICE)
            optimizer.zero_grad()
            logity = model(tokeny, dlugosci)
            loss = kryterium(logity, etykiety)
            loss.backward()  # W PyTorchu BPTT dzieje się automatycznie.
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            suma_strat += loss.item() * len(etykiety)
        strata_test, trafnosc_test = ocen(model, test_loader)
        historia["strata_trening"].append(suma_strat / len(trening))
        historia["strata_test"].append(strata_test)
        historia["trafnosc_test"].append(trafnosc_test)
    print(f"{typ:4s}: trafność testowa = {trafnosc_test:.1%}, "
          f"strata testowa = {strata_test:.3f}")
    return model, historia

### Eksperyment: porównanie trzech architektur

Uruchom komórkę. Zwróć uwagę, **czy** modele uczą się uwzględniać negację „nie”. Nie oceniaj modeli wyłącznie po jednej próbie na sztucznych danych – normalnie porównuje się wyniki na wielu losowych ziarnach i realistycznych zbiorach.

In [ ]:
modele = {}
historie = {}
for typ in ["RNN", "LSTM", "GRU"]:
    modele[typ], historie[typ] = trenuj(typ=typ, epoki=12)

fig, osie = plt.subplots(1, 2, figsize=(11, 3.5))
for typ, hist in historie.items():
    osie[0].plot(hist["strata_test"], marker=".", label=typ)
    osie[1].plot(hist["trafnosc_test"], marker=".", label=typ)
osie[0].set_title("Strata na zbiorze testowym")
osie[1].set_title("Trafność na zbiorze testowym")
for ax in osie:
    ax.set_xlabel("Epoka (od 0)")
    ax.grid(alpha=0.3)
    ax.legend()
osie[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 6. Sprawdź własne zdania

Wprowadź dowolną opinię, najlepiej korzystając ze znanych modelowi słów. Słowa spoza słownika zostaną oznaczone `<unk>` – to ograniczenie naszego uproszczonego przykładu. Zwróć uwagę, że w prawdziwym NLP model może łatwo pomylić się w bardziej subtelnych przypadkach.

In [ ]:
def przewidz_zdanie(tekst, model):
    model.eval()
    tokeny = torch.tensor([zakoduj(tekst)], dtype=torch.long, device=DEVICE)
    dlugosci = torch.tensor([tokeny.shape[1]], device=DEVICE)
    with torch.no_grad():
        logits = model(tokeny, dlugosci)
        p_pozytywna = torch.softmax(logits, dim=1)[0, 1].item()
    return p_pozytywna

for zdanie in ["ten film był świetny",
               "ten film był nie świetny",
               "moim zdaniem serial był fatalny",
               "moim zdaniem serial był nie fatalny"]:
    p = przewidz_zdanie(zdanie, modele["GRU"])
    print(f"{zdanie:45s} → P(pozytywna) = {p:.3f}")

### Ćwiczenia po wykładzie – NLP

**Zadanie A (łatwe):** Uruchom tylko `nn.RNN` oraz tylko `nn.LSTM`. Co zmienia jedna linijka w konstruktorze? Porównaj liczbę parametrów: `sum(p.numel() for p in model.parameters())`.

**Zadanie B (średnie):** Zmień `rozmiar_stanu=32` na 8, a następnie na 64. Kiedy model uczy się najłatwiej? Czy większy zawsze znaczy lepszy?

**Zadanie C (średnie):** Usuń `nn.utils.clip_grad_norm_` i porównaj przebieg treningu. **Brak widocznej różnicy jest poprawną obserwacją**, bo to zadanie jest łatwe i krótkie.

**Zadanie D (dodatkowe):** Dopisz nowy szablon zdań (np. „ten film wydawał mi się ...”) i własny słownik, po czym ponownie utwórz podział treningowy i model. Wymyśl przykłady, w których sama obecność słowa „dobry” prowadziłaby do pomyłki.

**Pytanie do dyskusji:** Dlaczego przypadkowy podział wierszy z tego *syntetycznego* zbioru jest znacznie łatwiejszy niż test na zdaniach napisanych przez nowych ludzi?

---
# Dodatek (opcjonalny): przewidywanie szeregu czasowego

RNN działa też na sekwencjach **liczb**, nie tylko słów. Tutaj model otrzyma 24 poprzednie wartości sygnału i spróbuje przewidzieć kolejną. To dodatek: **na wykładzie więcej czasu poświęcamy NLP**.

Nie pobieramy danych z internetu: generujemy sygnał podobny do okresowego pomiaru. Stosujemy **chronologiczny** podział trening/test, aby przyszłość nie przeciekała do uczenia. Porównujemy GRU z prostą regułą „następna wartość = ostatnia zaobserwowana”.

In [ ]:
# Sygnał o dwóch częstotliwościach i niewielkim poziomie szumu.
t = np.arange(600, dtype=np.float32)
sygnal = (np.sin(t * 0.13) + 0.25 * np.sin(t * 0.031)
          + 0.04 * np.random.default_rng(SEED).normal(size=len(t)))

granica_treningu = 420
okno = 24
srednia = sygnal[:granica_treningu].mean()
odchylenie = sygnal[:granica_treningu].std()
sygnal_norm = (sygnal - srednia) / odchylenie

X_train, y_train, X_test, y_test, indeksy_test = [], [], [], [], []
for idx_celu in range(okno, len(sygnal_norm)):
    fragment = sygnal_norm[idx_celu - okno:idx_celu]
    cel = sygnal_norm[idx_celu]
    if idx_celu < granica_treningu:
        X_train.append(fragment)
        y_train.append(cel)
    else:
        X_test.append(fragment)
        y_test.append(cel)
        indeksy_test.append(idx_celu)

# Tensor ma wymiary [przykłady, 24 kroki, 1 cecha].
X_train = torch.tensor(np.array(X_train), dtype=torch.float32).unsqueeze(-1)
y_train = torch.tensor(np.array(y_train), dtype=torch.float32).unsqueeze(-1)
X_test = torch.tensor(np.array(X_test), dtype=torch.float32).unsqueeze(-1)
y_test = torch.tensor(np.array(y_test), dtype=torch.float32).unsqueeze(-1)
print("Trening:", tuple(X_train.shape), "Test:", tuple(X_test.shape))

plt.figure(figsize=(10, 3))
plt.plot(t, sygnal)
plt.axvline(granica_treningu, linestyle="--", label="Początek testu")
plt.xlabel("Krok czasu")
plt.ylabel("Pomiar")
plt.legend()
plt.show()

In [ ]:
class PrognozaGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=24, batch_first=True)
        self.wyjscie = nn.Linear(24, 1)

    def forward(self, x):
        stany, _ = self.gru(x)
        return self.wyjscie(stany[:, -1, :])  # Tylko ostatni krok.


torch.manual_seed(SEED)
prognoza = PrognozaGRU().to(DEVICE)
optymalizator = torch.optim.Adam(prognoza.parameters(), lr=0.01)
strata_mse = nn.MSELoss()
loader_ts = DataLoader(torch.utils.data.TensorDataset(X_train, y_train),
                       batch_size=32, shuffle=True)

for epoka in range(30):
    prognoza.train()
    for xx, yy in loader_ts:
        xx, yy = xx.to(DEVICE), yy.to(DEVICE)
        optymalizator.zero_grad()
        blad = strata_mse(prognoza(xx), yy)
        blad.backward()
        nn.utils.clip_grad_norm_(prognoza.parameters(), 1.0)
        optymalizator.step()

prognoza.eval()
with torch.no_grad():
    przewidywania = prognoza(X_test.to(DEVICE)).cpu().squeeze(-1).numpy()
prawda = y_test.squeeze(-1).numpy()
# Naiwny model w prognozie jednokrokowej kopiuje ostatni punkt wejścia.
naiwny = X_test[:, -1, 0].numpy()

print("MSE GRU:   ", round(float(np.mean((przewidywania - prawda) ** 2)), 4))
print("MSE naiwny:", round(float(np.mean((naiwny - prawda) ** 2)), 4))

# Rysujemy tylko fragment zbioru testowego, żeby był czytelny.
fragment = slice(0, 80)
plt.figure(figsize=(11, 4))
plt.plot(np.array(indeksy_test)[fragment], prawda[fragment], label="Rzeczywistość", linewidth=2)
plt.plot(np.array(indeksy_test)[fragment], przewidywania[fragment], label="GRU")
plt.plot(np.array(indeksy_test)[fragment], naiwny[fragment], label="Prosta reguła: ostatnia wartość", alpha=0.7)
plt.xlabel("Krok czasu (zbiór testowy)")
plt.ylabel("Wartość znormalizowana")
plt.title("Prognoza jednego kroku naprzód")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Ćwiczenia dodatkowe – szeregi czasowe

1. Zmień `okno` z 24 na 8 i 48. Czy długość historii ma znaczenie?
2. Zamień `nn.GRU` na `nn.RNN` lub `nn.LSTM` (zwróć uwagę, że każda zwraca `(out, state)`; w LSTM drugi element jest parą stanów).
3. Spróbuj zwiększyć szum w sygnale. Czy prosty model naiwny może być równie dobry jak sieć?
4. Dlaczego **nie** należy mieszać losowo sąsiadujących próbek czasowych przed ustaleniem zbioru testowego?

**Wskazówka metodyczna:** przy prognozowaniu jednego kroku można na teście używać prawdziwych wcześniejszych pomiarów. Prognoza 20 kroków naprzód bez nowych obserwacji to **inny i trudniejszy problem**.

---
## Odpowiedzi / wskazówki do najważniejszych pytań

- **Dlaczego ten sam zestaw wag?** Dzięki temu jedna sieć obsługuje wiele kroków i długości sekwencji, zamiast mieć osobną sieć dla każdego położenia.
- **Po co LSTM / GRU?** Ich bramki ułatwiają kontrolowanie przepływu i zapamiętywania informacji w długich sekwencjach. Nie gwarantują perfekcyjnej pamięci.
- **BPTT a zwykły backprop?** To ta sama zasada różniczkowania złożonej funkcji, ale sieć jest rozwinięta na kolejne kroki czasu.
- **Czy GRU jest zawsze lepszy od LSTM?** Nie. Ma mniej bramek i zwykle prostszą budowę, lecz jakość zależy od danych, budżetu i strojenia hiperparametrów.
- **Dlaczego zbiór testowy musi być oddzielony?** Wynik na danych użytych do nauki nie mówi, czy model dobrze uogólnia na nowe przypadki.